## 🎯 Learning Objectives
* Understand the role and importance of an intent-routing agent in a multi-agent AI system.
* Learn how to design and implement a basic intent-routing agent using a Large Language Model (LLM).
* Analyze the performance considerations, trade-offs, and typical use cases for intent routing.


## Implementing the Intent-Routing Agent: The AI System's Traffic Controller

In the intricate world of multi-agent AI systems, especially for complex tasks like hotel reservations, a crucial component is the **intent-routing agent**. Imagine a bustling airport control tower, where air traffic controllers direct incoming and outgoing flights to their designated runways and gates. Without this central coordination, chaos would ensue, leading to delays, collisions, and frustrated passengers.

Similarly, an intent-routing agent acts as the central 'traffic controller' or 'receptionist' for your AI system. Its primary job is to listen to a user's request, understand their underlying *intent*, and then intelligently direct that request to the most appropriate specialized agent within the system. For our hotel reservation project, this means distinguishing between a user wanting to `book_room`, `check_availability`, `modify_booking`, `cancel_booking`, or simply ask a `general_inquiry`.

### Why is Intent Routing Essential?

1.  **Efficiency**: Instead of every agent trying to understand every query, the router ensures only the relevant agent processes the request, saving computational resources and time.
2.  **Scalability**: As your system grows with more specialized agents (e.g., a 'loyalty program agent', a 'restaurant booking agent'), the router can easily incorporate new routing rules without overhauling existing agents.
3.  **Clarity and Focus**: Each specialized agent can be designed to excel at a single task, without being burdened by the complexity of understanding diverse user intents.
4.  **Improved User Experience**: Users get faster, more accurate responses because their queries are handled by the most competent part of the system.

### How Does It Work?

The core of an intent-routing agent typically involves a powerful Large Language Model (LLM) acting as the 'brain'. Here's a step-by-step breakdown:

1.  **User Input**: A user provides a natural language query (e.g., "I want to book a room for two nights next month.").
2.  **Intent Classification**: The intent-routing agent sends this query to an LLM, along with a predefined list of possible intents. The LLM's task is to classify the user's query into one of these known intents.
3.  **Routing Logic**: Based on the LLM's classification, the router applies a simple logic to determine which specialized agent should handle the request. For instance, if the intent is `book_room`, it routes to the `Booking Agent`.
4.  **Delegation**: The user's original query (or a refined version of it) is then passed to the designated specialized agent for further processing.

In this lesson, we'll implement a foundational intent-routing agent using a modern LLM, demonstrating how it can effectively categorize user requests and prepare them for delegation within our multi-agent hotel reservation system.


In [ ]:
# Ensure you have the Google Generative AI client library installed:
# pip install google-generativeai

import os
import google.generativeai as genai

# --- Configuration ---
# It's crucial to set your Google API key as an environment variable.
# For example, in your terminal:
# export GOOGLE_API_KEY='YOUR_API_KEY'
# Or, if running in a Jupyter environment, you might set it directly (not recommended for production):
# os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

# Check if the API key is set
if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY environment variable not set. Please set it before running the code.")

# Configure the generative AI model
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Initialize the Gemini Pro 1.5 model (or a suitable alternative for 2026)
# Gemini 1.5 Pro is chosen for its strong reasoning and context window capabilities.
model = genai.GenerativeModel('gemini-1.5-pro')

# --- Define Available Intents ---
# These are the specific actions our multi-agent system can handle.
# Each intent should ideally map to a specialized agent.
AVAILABLE_INTENTS = [
    "book_room",
    "check_availability",
    "modify_booking",
    "cancel_booking",
    "general_inquiry",
    "unrecognized_intent" # Fallback for queries that don't fit any category
]

# --- Intent Routing Agent Implementation ---
class IntentRouterAgent:
    def __init__(self, model, intents):
        self.model = model
        self.intents = intents
        self.intent_list_str = ", ".join(intents)

    def classify_intent(self, user_query: str) -> str:
        """
        Classifies the user's query into one of the predefined intents using an LLM.
        """
        prompt = f"""
        You are an intent classification system for a hotel reservation service.
        Your task is to identify the primary intent of the user's query from the following list:
        [{self.intent_list_str}]

        Respond with ONLY the intent name, exactly as it appears in the list. Do not add any other text or punctuation.
        If the query does not clearly match any of the listed intents, classify it as 'unrecognized_intent'.

        User Query: "{user_query}"
        Intent:
        """

        try:
            # Use the LLM to generate the intent classification
            # Setting temperature to a low value (e.g., 0.1) encourages more deterministic output.
            response = self.model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(temperature=0.1)
            )
            # Extract the text from the response
            classified_intent = response.text.strip()

            # Basic validation: ensure the classified intent is one of our known intents
            if classified_intent not in self.intents:
                print(f"Warning: LLM returned an unexpected intent '{classified_intent}'. Defaulting to 'unrecognized_intent'.")
                return "unrecognized_intent"

            return classified_intent
        except Exception as e:
            print(f"Error during intent classification: {e}")
            return "unrecognized_intent" # Fallback in case of API errors

# --- Example Usage ---
if __name__ == "__main__":
    router = IntentRouterAgent(model, AVAILABLE_INTENTS)

    test_queries = [
        "I'd like to reserve a room for next weekend.",
        "Do you have any rooms available from July 10th to July 15th?",
        "Can I change my reservation for John Doe?",
        "I need to cancel my booking under confirmation number XYZ123.",
        "What are your check-in and check-out times?",
        "Tell me a joke.",
        "I want to book a room for 3 nights starting tomorrow.",
        "Is there a gym at the hotel?",
        "My booking reference is ABC456, I need to cancel it."
    ]

    print("--- Intent Classification Results ---")
    for i, query in enumerate(test_queries):
        print(f"\nQuery {i+1}: '{query}'")
        intent = router.classify_intent(query)
        print(f"Classified Intent: {intent}")

        # In a real multi-agent system, you would now route to the specific agent:
        # if intent == "book_room":
        #     booking_agent.handle_request(query)
        # elif intent == "check_availability":
        #     availability_agent.handle_request(query)
        # # ... and so on for other intents
        # else:
        #     fallback_agent.handle_request(query)


### Interpreting the Output and Performance Considerations

The code above demonstrates a functional intent-routing agent. When you run it, you'll see each test query classified into one of the `AVAILABLE_INTENTS`. For example, a query like "I'd like to reserve a room for next weekend" should be correctly classified as `book_room`, while "Tell me a joke" would likely fall under `unrecognized_intent` or `general_inquiry` depending on the LLM's interpretation and the prompt's strictness.

#### Performance Trade-offs:

1.  **Accuracy**: The precision of intent classification heavily relies on the underlying LLM's capabilities and the quality of your prompt. A well-defined list of intents and clear instructions to the LLM (e.g., "respond with ONLY the intent name") are crucial. For production systems, you might consider:
    *   **Few-shot prompting**: Providing examples of queries and their correct intents within the prompt.
    *   **Fine-tuning**: For highly specific or nuanced domains, fine-tuning a smaller LLM on your own dataset of queries and intents can yield superior accuracy and potentially lower costs.
    *   **Confidence Scores**: Some LLM APIs provide confidence scores for their classifications, allowing you to set thresholds for routing or escalating to human review.

2.  **Latency**: Each call to the LLM API introduces network latency and processing time. For real-time applications, this can be a bottleneck. Strategies to mitigate this include:
    *   **Asynchronous calls**: Processing multiple queries concurrently.
    *   **Caching**: For frequently asked or identical queries.
    *   **Local models**: Using smaller, optimized models deployed locally or on edge devices for faster inference, though these might sacrifice some accuracy compared to large cloud models.

3.  **Cost**: LLM API usage is typically billed per token. A high volume of requests can lead to significant costs. Optimizations include:
    *   **Prompt engineering**: Making prompts concise without losing necessary context.
    *   **Batching requests**: If the API supports it, sending multiple queries in a single request.
    *   **Model selection**: Using smaller, more cost-effective models for simpler classification tasks if they meet accuracy requirements.

4.  **Robustness and Fallbacks**: What happens if the LLM misclassifies an intent or fails to respond? Our example includes a `try-except` block and a fallback `unrecognized_intent`. In a production system, you'd want more sophisticated error handling, potentially involving:
    *   **Human-in-the-loop**: Routing ambiguous or `unrecognized_intent` queries to a human agent for review.
    *   **Retry mechanisms**: For transient API errors.
    *   **Default actions**: For critical functions, having a safe default if classification fails.

#### Typical Use Cases:

Beyond hotel reservations, intent-routing agents are fundamental in:

*   **Customer Service Chatbots**: Directing user queries to sales, support, billing, or technical assistance departments.
*   **Task Automation**: Identifying user commands for smart home devices, virtual assistants, or workflow automation tools.
*   **Content Moderation**: Classifying user-generated content for policy violations.
*   **Data Entry and Processing**: Categorizing incoming documents or emails for automated processing.
*   **Educational Platforms**: Routing student questions to relevant course modules or tutors.

By mastering intent routing, you lay a robust foundation for building intelligent, scalable, and efficient multi-agent AI systems.


### Resources for Further Learning

*   **Google Gemini API Documentation**: Explore the full capabilities of Google's generative AI models for more advanced prompting and integration.
    *   [Google AI Studio](https://aistudio.google.com/)
    *   [Gemini API Documentation](https://ai.google.dev/docs)

*   **Hugging Face Transformers**: For understanding and potentially fine-tuning smaller, open-source LLMs for intent classification.
    *   [Hugging Face Transformers Library](https://huggingface.co/docs/transformers/index)
    *   [Hugging Face Models](https://huggingface.co/models)

*   **LangChain / LlamaIndex**: Frameworks that provide higher-level abstractions for building LLM-powered applications, including agent orchestration and routing.
    *   [LangChain Documentation](https://www.langchain.com/)
    *   [LlamaIndex Documentation](https://www.llamaindex.ai/)

*   **CrewAI**: A framework specifically designed for building multi-agent systems, which often incorporates intent routing as a core component.
    *   [CrewAI Documentation](https://www.crewai.com/)

*   **Prompt Engineering Guides**: Learn best practices for crafting effective prompts to maximize LLM performance.
    *   [Google's Prompt Engineering Guide](https://ai.google.dev/docs/prompt_engineering_guidelines)
    *   [OpenAI's Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
